# D02 — 삼성전자 공시 목록과 정정 접수번호 조회

실행 흐름: 조회 조건 설정 → OpenDART 목록 조회 → 원본 행에 정정 접수번호 연결 → JSON 저장.

정정 접수번호는 원본 공시 행에만 기록합니다. 정정 제출 행은 원본 행의 후속 정정 후보로 사용합니다. API가 원 공시와 정정본 사이의 직접 연결 키를 제공하지 않아 회사코드, 정리한 보고서명, 접수 순서로 후보를 찾습니다. 후보가 여러 건이거나 제목이 다르면 DART 원문에서 확인하세요. 인증키는 출력하거나 저장하지 않습니다.

In [26]:
from pathlib import Path
from collections import defaultdict
import json
import os
import re
import time
import urllib.parse
import urllib.request

# API와 파일
API_URL = 'https://opendart.fss.or.kr/api/list.json'
USER_AGENT = 'D02-disclosure-list-check/1.0'
PROJECT_DIR = next(
    (folder for folder in (Path.cwd(), *Path.cwd().parents)
     if (folder / 'dart_results' / 'company_stock_005930.json').is_file()),
    Path.cwd(),
)
COMPANY_FILE = PROJECT_DIR / 'dart_results' / 'company_stock_005930.json'
OUTPUT_FILE = PROJECT_DIR / 'dart_results' / 'disclosures_list_stock_005930.json'

# 조회 조건
QUERY = {
    'bgn_de': '20260101',
    'end_de': '20261012',
    'last_reprt_at': 'N',  # 정정 보고서를 포함한 제출 보고서 전체
    'sort': 'date',
    'sort_mth': 'desc',
}
SAVE_COUNT = 200# None이면 기간 전체, 숫자면 최신 해당 건수만 조회·저장
REQUEST_PAUSE = 0.2
if SAVE_COUNT is not None and SAVE_COUNT < 1:
    raise ValueError('SAVE_COUNT는 1 이상의 정수 또는 None이어야 합니다.')
API_PAGE_COUNT = 100 if SAVE_COUNT is None else min(100, SAVE_COUNT)

CORRECTION_TYPES = ('기재정정', '첨부정정', '첨부추가', '발행조건확정', '연장결정', '정정명령부과', '정정제출요구')
CORRECTION_LABELS = tuple(f'[{kind}]' for kind in CORRECTION_TYPES)
CORRECTION_PREFIX = re.compile(
    r'^\s*(?:' + '|'.join(re.escape(x) for x in CORRECTION_LABELS) + r')\s*'
)

print('작업 폴더:', Path.cwd())


작업 폴더: C:\ai_campus\proj


In [27]:
def read_env_value(env_file: Path, variable: str) -> str | None:
    # .env에서 변수 하나를 읽습니다.
    for line in env_file.read_text(encoding='utf-8-sig').splitlines():
        line = line.strip()
        if not line or line.startswith('#') or '=' not in line:
            continue
        name, value = line.split('=', 1)
        if name.strip() == variable:
            return value.strip().strip('\"\'')
    return None


def load_api_key() -> str:
    # 환경변수를 먼저 보고, 없으면 현재 폴더부터 상위 폴더까지 .env를 찾습니다.
    key = os.environ.get('DART_API_KEY')
    if key:
        return key

    for folder in (Path.cwd(), *Path.cwd().parents):
        env_file = folder / '.env'
        if env_file.is_file():
            key = read_env_value(env_file, 'DART_API_KEY')
            if key:
                return key

    raise RuntimeError('DART_API_KEY를 환경변수 또는 프로젝트 .env에 설정하세요.')


def load_company(path: Path) -> dict:
    # D01에서 저장한 기업정보를 불러옵니다.
    if not path.is_file():
        raise FileNotFoundError(f'기업정보 파일을 찾을 수 없습니다: {path}')
    return json.loads(path.read_text(encoding='utf-8'))


api_key = load_api_key()
company = load_company(COMPANY_FILE)
corp_code = company['corp_code']
print(f"조회 기업: {company['corp_name']} / corp_code={corp_code}")


조회 기업: 삼성전자 / corp_code=00126380


In [28]:
def get_disclosure_page(page_no: int) -> tuple[int, dict]:
    # 공시검색 API를 GET으로 한 페이지 조회합니다.
    params = {
        'crtfc_key': api_key,
        'corp_code': corp_code,
        **QUERY,
        'page_no': str(page_no),
        'page_count': str(API_PAGE_COUNT),  # SAVE_COUNT에서 자동 계산
    }
    url = f'{API_URL}?{urllib.parse.urlencode(params)}'
    request = urllib.request.Request(
        url, headers={'User-Agent': USER_AGENT}, method='GET'
    )

    try:
        with urllib.request.urlopen(request, timeout=60) as response:
            return response.status, json.loads(response.read().decode('utf-8'))
    except Exception as exc:
        # 예외 메시지에 인증키가 든 URL이 노출되지 않게 합니다.
        raise RuntimeError(f'OpenDART 요청 실패 ({type(exc).__name__})') from None


def get_disclosures() -> tuple[int, dict, list[dict]]:
    # SAVE_COUNT가 None이면 전체 페이지, 숫자면 필요한 페이지만 요청합니다.
    http_status, first_page = get_disclosure_page(page_no=1)
    if first_page.get('status') not in ('000', '013'):
        raise RuntimeError(
            f"OpenDART 오류 {first_page.get('status')}: {first_page.get('message')}"
        )

    rows = list(first_page.get('list', []))
    total_pages = int(first_page.get('total_page') or 1)
    pages_needed = total_pages if SAVE_COUNT is None else (SAVE_COUNT + API_PAGE_COUNT - 1) // API_PAGE_COUNT

    for page_no in range(2, min(total_pages, pages_needed) + 1):
        time.sleep(REQUEST_PAUSE)
        _, page = get_disclosure_page(page_no)
        if page.get('status') == '013':
            break
        if page.get('status') != '000':
            raise RuntimeError(
                f"OpenDART 오류 {page.get('status')}: {page.get('message')}"
            )
        rows.extend(page.get('list', []))

    if SAVE_COUNT is not None:
        rows = rows[:SAVE_COUNT]
    return http_status, first_page, rows

http_status, api_result, raw_rows = get_disclosures()
print(f"HTTP {http_status} / API {api_result.get('status')} ({api_result.get('message')})")
print(f"전체 {api_result.get('total_count')}건 중 {len(raw_rows)}건 확보")


HTTP 200 / API 000 (정상)
전체 2777건 중 200건 확보


In [29]:
def clean_report_title(title: str) -> str:
    # 제목 맨 앞의 정정 표기를 제거해 원 공시와 비교합니다.
    title = title.strip()
    while True:
        cleaned = CORRECTION_PREFIX.sub('', title, count=1).strip()
        if cleaned == title:
            return title
        title = cleaned


def prepare_disclosures(rows: list[dict]) -> list[dict]:
    # API 응답에서 쓸 필드를 추리고 정정 접수번호를 연결합니다.
    items = []
    for row in rows:
        title = row.get('report_nm') or ''
        receipt_no = row.get('rcept_no') or ''
        correction_type = next(
            (kind for kind in CORRECTION_TYPES if title.lstrip().startswith(f'[{kind}]') or title.lstrip().startswith(kind)),
            None,
        )
        is_correction = correction_type is not None
        items.append({
            'corp_cls': row.get('corp_cls', ''),
            'corp_name': row.get('corp_name', ''),
            'corp_code': row.get('corp_code', ''),
            'stock_code': row.get('stock_code', ''),
            'report_nm': title,
            'rcept_no': receipt_no,
            'flr_nm': row.get('flr_nm', ''),
            'rcept_dt': row.get('rcept_dt', ''),
            'rm': row.get('rm', ''),
            'is_correction_submission': is_correction,
            'correction_type': correction_type,
            'correction_followup_flag': '정' in (row.get('rm') or ''),
            'candidate_correction_rcept_no': None,
            'viewer_url': (
                f'https://dart.fss.or.kr/dsaf001/main.do?rcpNo={receipt_no}'
                if receipt_no else None
            ),
        })

    # 정정 제출을 회사코드와 정리한 보고서명으로 모읍니다.
    corrections = defaultdict(list)
    for item in items:
        if item['is_correction_submission']:
            key = (item['corp_code'], clean_report_title(item['report_nm']))
            corrections[key].append(item)

    # 원 공시에 정정 안내가 있으면 그 뒤에 접수된 정정 후보를 기록합니다.
    for item in items:
        # 이 열은 원본 행에만 채웁니다. 정정 제출 행에는 자기 번호를 넣지 않습니다.
        if item['is_correction_submission']:
            continue
        if not item['correction_followup_flag']:
            continue

        key = (item['corp_code'], clean_report_title(item['report_nm']))
        original_order = (item['rcept_dt'], item['rcept_no'])
        receipt_numbers = [
            candidate['rcept_no']
            for candidate in corrections.get(key, [])
            if (candidate['rcept_dt'], candidate['rcept_no']) > original_order
        ]
        item['candidate_correction_rcept_no'] = ', '.join(receipt_numbers) or None

    return items


all_disclosures = prepare_disclosures(raw_rows)
recent_disclosures = all_disclosures[:50]

print('접수일 | 보고서명 | 접수번호 | 정정 여부 | 정정 유형 | 정정 접수번호')
for item in recent_disclosures:
    print(
        f"{item['rcept_dt']} | {item['report_nm']} | {item['rcept_no']} | "
        f"{True if item['is_correction_submission'] else False} | "
        f"{item['correction_type'] or None} | {item['candidate_correction_rcept_no'] or None}"
    )

접수일 | 보고서명 | 접수번호 | 정정 여부 | 정정 유형 | 정정 접수번호
20261012 | 자기주식취득결과보고서 | 20261012000421 | False | None | None
20261012 | 임원ㆍ주요주주특정증권등소유상황보고서 | 20261012000021 | False | None | None
20261008 | 주식등의대량보유상황보고서(일반) | 20261008000706 | False | None | None
20261008 | 기업설명회(IR)개최(안내공시)               | 20261008800028 | False | None | None
20261008 | 연결재무제표기준영업(잠정)실적(공정공시)               | 20261008800004 | False | None | None
20261002 | 최대주주등소유주식변동신고서               | 20261002800981 | False | None | None
20261001 | 임원ㆍ주요주주특정증권등소유상황보고서 | 20261001000251 | False | None | None
20261001 | 임원ㆍ주요주주특정증권등소유상황보고서 | 20261001000203 | False | None | None
20260922 | 임원ㆍ주요주주특정증권등소유상황보고서 | 20260922000524 | False | None | None
20260918 | 최대주주등소유주식변동신고서               | 20260918800754 | False | None | None
20260918 | 임원ㆍ주요주주특정증권등소유상황보고서 | 20260918000212 | False | None | None
20260918 | 임원ㆍ주요주주특정증권등소유상황보고서 | 20260918000191 | False | None | None
20260917 | 임원ㆍ주요주주특정증권등소유상황보고서 | 20260917000097 | False | None | None
20260914 

In [30]:
# 인증키는 제외하고 조회 조건과 결과만 JSON으로 저장합니다.
sample = {
    'query': {
        'corp_code': corp_code,
        'corp_name': company.get('corp_name'),
        **QUERY,
    },
    'http_status': http_status,
    'api_status': api_result.get('status'),
    'api_message': api_result.get('message'),
    'total_count': api_result.get('total_count'),
    'fetched_count': len(all_disclosures),
    'fields': list(raw_rows[0].keys()) if raw_rows else [],
    'save_count': len(all_disclosures),
    'disclosures': all_disclosures,
}

OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
OUTPUT_FILE.write_text(
    json.dumps(sample, ensure_ascii=False, indent=2) + '\n',
    encoding='utf-8',
)
print('JSON 덮어쓰기 완료:', OUTPUT_FILE.resolve())

JSON 덮어쓰기 완료: C:\ai_campus\proj\dart_results\disclosures_list_stock_005930.json


## 결과 해석

- 출력 표는 `접수일 | 보고서명 | 접수번호 | 정정 여부 | 정정 유형 | 정정 접수번호` 순서입니다.
- `정정 여부`와 `정정 유형`은 보고서명에 표시된 유형으로 판별합니다. 해당하지 않으면 `아니오`, 유형은 `-`로 표시됩니다.
- `candidate_correction_rcept_no`는 원본 공시 행에만 채웁니다. 원본의 `rm`에 `정`이 있으면 조회 기간 전체에서 같은 회사·정리된 제목·이후 접수 순서에 맞는 정정 제출을 찾아 번호를 기록합니다.
- API에 직접 연결 키가 없어 제목이 바뀐 공시는 자동으로 연결되지 않을 수 있습니다. `viewer_url`에서 DART 원문을 확인하세요.
- `SAVE_COUNT = None`이면 해당 기간 전체를 조회·저장하고, 숫자를 지정하면 최신 해당 건수만 조회·저장합니다. 페이지당 최대 100건이며 필요한 페이지만 요청합니다.